# 🧠 Day 5: Independent Lab — From Predictions to Decisions

## Your mission

You already know how to build churn + value models. Now you will make the solution **decision-ready**.

### Minimum deliverables
1. Improved churn report (ROC-AUC, PR-AUC, lift@10%)
2. Improved value model report (MAE/RMSE)
3. Final Revenue-at-Risk call list (CSV)
4. A short manager recommendation (bullets)

### Choose ONE extension track (A/B/C/D)
- **A:** cost & capacity targeting (recommended)
- **B:** calibration & probability quality
- **C:** AutoML benchmark (FLAML)
- **D:** segment stress test

---
## Part 0: Setup (5 min)

In [ ]:
!pip install -q -U pandas numpy scikit-learn shap google-genai
# Track C AutoML (optional)
# !pip install -q -U flaml

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timezone

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, accuracy_score,
    mean_absolute_error, mean_squared_error, r2_score,
    brier_score_loss
)

import shap
shap.initjs()

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

### GenAI Setup

In [ ]:
# ── Google GenAI Setup ───────────────────────────────────
import google.generativeai as genai

try:
    from google.colab import userdata
    API_KEY = userdata.get("GOOGLE_API_KEY")
except ImportError:
    import getpass
    API_KEY = getpass.getpass("Enter your Google API key: ")

genai.configure(api_key=API_KEY)

# Model configuration
MODEL_ID = "gemini-2.5-flash-lite"
client = genai.client

print(f"✅ GenAI configured with model: {MODEL_ID}")

In [ ]:
# ── Prompt Logging Infrastructure ───────────────────────
PROMPT_LOG = []

def log_interaction(prompt, response_text, model=MODEL_ID):
    """Log a GenAI interaction to PROMPT_LOG."""
    entry = {
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "model": model,
        "prompt": prompt[:200] + "..." if len(prompt) > 200 else prompt,
        "response_snippet": response_text[:300] + "..." if len(response_text) > 300 else response_text,
    }
    PROMPT_LOG.append(entry)
    return entry

def show_log(limit=None):
    """Display the prompt log."""
    log_df = pd.DataFrame(PROMPT_LOG)
    if limit:
        log_df = log_df.tail(limit)
    return log_df

print("✅ Logging infrastructure ready")

---
## Part 1: Load data + build a strong baseline (15–20 min)

✅ **Task:** Copy your working solution from the Guided Lab (or re-run it here).

**You should end this part with:**
- best churn model (manual)
- best regression model (manual)
- baseline Revenue-at-Risk call list

### TODO: Load Telco dataset and rebuild your baseline models

Hints:
- Use the same URL as in the guided lab
- Clean `TotalCharges`
- Create `y_clf` and `y_reg`
- Build preprocessing pipelines
- Fit logit/RF/GB for churn; ridge/RFR/GBR for value

In [ ]:
# ── TODO: Load data ─────────────────────────────────────
url = "https://raw.githubusercontent.com/blastchar/telco-customer-churn/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip().replace("", np.nan), errors="coerce")
customer_ids = df["customerID"].copy()

y_clf = (df["Churn"] == "Yes").astype(int)
y_reg = df["MonthlyCharges"].astype(float)
X = df.drop(columns=["customerID", "Churn"])

In [ ]:
# ── TODO: Train/test split (stratify for churn) ────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y_clf,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_clf
)
yreg_train = y_reg.loc[X_train.index]
yreg_test = y_reg.loc[X_test.index]

### TODO: Define preprocessing + fit 3 churn models

You can start with the same defaults as the guided lab and then improve them.

In [ ]:
# ── TODO: Preprocessing ──────────────────────────────────
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([( "impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()) ]), numeric_cols),
        ("cat", Pipeline([( "impute", SimpleImputer(strategy="most_frequent")), ("onehot", ohe) ]), categorical_cols),
    ]
)

In [ ]:
def evaluate_classifier(model, X_te, y_te, threshold=0.5):
    """Evaluate a classifier and return metrics + probabilities."""
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_te, proba),
        "pr_auc": average_precision_score(y_te, proba),
        "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred, zero_division=0),
        "accuracy": accuracy_score(y_te, pred),
    }, proba


def eval_regression(model, X_te, y_te):
    """Evaluate a regression model."""
    pred = model.predict(X_te)
    return {
        "mae": mean_absolute_error(y_te, pred),
        "rmse": mean_squared_error(y_te, pred, squared=False),
        "r2": r2_score(y_te, pred)
    }, pred


def lift_by_decile(y_true, y_score, n_bins=10):
    """Compute lift by decile (manager-friendly ranking metric)."""
    tmp = pd.DataFrame({"y": y_true, "score": y_score}).copy()
    tmp["decile"] = pd.qcut(tmp["score"].rank(method="first"), q=n_bins, labels=False) + 1
    overall = tmp["y"].mean()
    table = (
        tmp.groupby("decile")
           .agg(n=("y", "size"), churn_rate=("y", "mean"), avg_score=("score", "mean"))
           .sort_index(ascending=False)
           .reset_index()
    )
    table["lift"] = table["churn_rate"] / overall
    return table

print("✅ Helper functions defined.")

In [ ]:
# ── TODO: Fit churn models ──────────────────────────────
logit = Pipeline([( "prep", preprocess), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")) ])
rf = Pipeline([( "prep", preprocess), ("clf", RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=RANDOM_STATE, class_weight="balanced_subsample")) ])
gb = Pipeline([( "prep", preprocess), ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE)) ])

for m in [logit, rf, gb]:
    m.fit(X_train, y_train)

m_logit, p_logit = evaluate_classifier(logit, X_test, y_test)
m_rf, p_rf = evaluate_classifier(rf, X_test, y_test)
m_gb, p_gb = evaluate_classifier(gb, X_test, y_test)

pd.DataFrame([
    {"model": "logit", **m_logit},
    {"model": "rf", **m_rf},
    {"model": "gb", **m_gb},
]).sort_values(["pr_auc", "roc_auc"], ascending=False)

### GenAI: Ask for feature engineering suggestions

In [ ]:
# ── TODO (GenAI): Ask for feature engineering ideas ────
# Uncomment and customize this section to ask GenAI for feature suggestions

# feature_prompt = f"""
# I have trained churn models on a telco dataset with these features:
# {list(X_train.columns)}
# 
# Current best model: Random Forest with PR-AUC={m_rf['pr_auc']:.3f}
# 
# Suggest 3-5 NEW features I could engineer to improve churn prediction.
# For each feature, provide:
# 1. Feature name
# 2. Business intuition (why it matters for churn)
# 3. Pandas code to create it
# """

# response = client.models.generate_content(model=MODEL_ID, contents=feature_prompt)
# print(response.text)
# log_interaction(feature_prompt, response.text)

print("⏳ TODO (optional): Send feature engineering prompt to GenAI")

---
## Part 2: Choose ONE extension track (60–80 min)

Pick your track and complete the TODO sections below.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  SELECT YOUR TRACK — change the letter below            ║
# ╚══════════════════════════════════════════════════════════╝
SELECTED_TRACK = ""   # Change to "A", "B", "C", or "D"

assert SELECTED_TRACK in ("A", "B", "C", "D"), "⚠️ Set SELECTED_TRACK to 'A', 'B', 'C', or 'D'"
print(f"✅ Selected Track: {SELECTED_TRACK}")

### Track A — Cost & Capacity Targeting (recommended)

✅ **Goal:** pick a threshold and/or top-N rule that reflects your business reality.

**Inputs (choose your own numbers):**
- offer cost $c$ (e.g., 10€)
- churn loss $L$ (e.g., 150€)
- capacity: top N calls/week (e.g., 500)

**Deliverables:**
- chosen rule (threshold or top-N)
- expected value calculation (simple is fine)

In [ ]:
# ── TODO (Track A): define costs and compute a cost-based threshold
if SELECTED_TRACK == "A":
    c = 10      # offer cost
    L = 150     # churn loss
    threshold_cost = c / L
    print(f"Cost-based threshold: {threshold_cost:.3f}")
else:
    print("⏭️ Skipping Track A")

In [ ]:
# ── TODO (Track A): compare outcomes at different thresholds
if SELECTED_TRACK == "A":
    def expected_value(y_true, p, threshold, c, L):
        pred = (p >= threshold).astype(int)
        # contact cost for predicted positives
        cost = pred.sum() * c
        # churn loss for missed churners (false negatives)
        fn = ((y_true == 1) & (pred == 0)).sum()
        loss = fn * L
        return -(cost + loss), pred.sum(), fn

    p_use = p_rf  # ── TODO: set to your chosen model probabilities ───────
    for t in [0.05, 0.10, 0.20, threshold_cost, 0.50]:
        val, n_contact, fn = expected_value(y_test.values, p_use, t, c, L)
        print(f"t={t:.3f} | contacts={n_contact:4d} | FN={fn:4d} | -cost-loss={val:,.0f}")
else:
    print("⏭️ Skipping Track A")

### Track B — Calibration & Probability Quality

✅ **Goal:** check whether predicted probabilities mean what they claim.

**Deliverables:**
- calibration curve
- Brier score
- calibrated model (sigmoid or isotonic)

In [ ]:
# ── TODO (Track B): Calibration ──────────────────────────
if SELECTED_TRACK == "B":
    from sklearn.calibration import calibration_curve, CalibratedClassifierCV

    # ── TODO: Choose a base model ────────────────────────
    base_model = rf  # Change to logit or gb if you prefer

    cal = CalibratedClassifierCV(base_model, method="sigmoid", cv=3)
    cal.fit(X_train, y_train)

    p_cal = cal.predict_proba(X_test)[:, 1]
    brier = brier_score_loss(y_test, p_cal)
    print(f"Brier score (calibrated): {brier:.4f}")
else:
    print("⏭️ Skipping Track B")

In [ ]:
# ── TODO (Track B): Plot calibration curve ──────────────
if SELECTED_TRACK == "B":
    prob_true, prob_pred = calibration_curve(y_test, p_cal, n_bins=10)
    plt.figure(figsize=(5, 5))
    plt.plot(prob_pred, prob_true, marker="o")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed churn rate")
    plt.title("Calibration curve")
    plt.show()
else:
    print("⏭️ Skipping Track B")

### Track C — AutoML Benchmark (FLAML)

✅ **Goal:** run AutoML as a benchmark and compare on the same holdout set.

**Deliverables:**
- churn: ROC-AUC + PR-AUC vs. your manual best model
- regression: MAE/RMSE vs. your manual best model
- recommendation: would you ship AutoML? why/why not?

In [ ]:
# ── TODO (Track C): install and run AutoML ──────────────
if SELECTED_TRACK == "C":
    # !pip install -q -U flaml
    # from flaml import AutoML
    #
    # automl = AutoML()
    # automl.fit(X_train, y_train, task="classification", metric="auc", time_budget=60)
    # p_automl = automl.predict_proba(X_test)[:, 1]
    # print("AutoML model:", automl.model)
    # print("ROC-AUC:", roc_auc_score(y_test, p_automl))
    # print("PR-AUC:", average_precision_score(y_test, p_automl))
    print("⏳ TODO (Track C): Uncomment and run FLAML AutoML")
else:
    print("⏭️ Skipping Track C")

### Track D — Segment Stress Test

✅ **Goal:** check whether performance is consistent across key segments.

**Deliverables:**
- at least 2 segments (e.g., Contract, tenure band)
- metric table by segment (PR-AUC or lift@10%)
- risk notes (where might the model fail?)

In [ ]:
# ── TODO (Track D): segment evaluation helper ───────────
if SELECTED_TRACK == "D":
    def segment_report(df_features, y_true, p, segment_col):
        tmp = df_features[[segment_col]].copy()
        tmp["y"] = y_true.values
        tmp["p"] = p
        out = []
        for g, part in tmp.groupby(segment_col):
            if part["y"].nunique() < 2:
                continue
            out.append({
                segment_col: g,
                "n": len(part),
                "churn_rate": part["y"].mean(),
                "roc_auc": roc_auc_score(part["y"], part["p"]),
                "pr_auc": average_precision_score(part["y"], part["p"]),
            })
        return pd.DataFrame(out).sort_values("pr_auc")

    print(segment_report(X_test, y_test, p_rf, "Contract"))
else:
    print("⏭️ Skipping Track D")

---
## Part 3: Final Revenue-at-Risk call list (15–20 min)

✅ **Task:** rebuild your MonthlyCharges model and create a final call list.

Deliverable: `day5_independent_call_list.csv`

In [ ]:
# ── TODO: Regression pipeline (drop MonthlyCharges from features) ─
Xr_train = X_train.drop(columns=["MonthlyCharges"])
Xr_test = X_test.drop(columns=["MonthlyCharges"])

num_r = Xr_train.select_dtypes(include=["number"]).columns.tolist()
cat_r = Xr_train.select_dtypes(exclude=["number"]).columns.tolist()

preprocess_r = ColumnTransformer(
    transformers=[
        ("num", Pipeline([( "impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()) ]), num_r),
        ("cat", Pipeline([( "impute", SimpleImputer(strategy="most_frequent")), ("onehot", ohe) ]), cat_r),
    ]
)

ridge = Pipeline([( "prep", preprocess_r), ("reg", Ridge(alpha=1.0)) ])
gbr = Pipeline([( "prep", preprocess_r), ("reg", GradientBoostingRegressor(random_state=RANDOM_STATE)) ])
ridge.fit(Xr_train, yreg_train)
gbr.fit(Xr_train, yreg_train)

pred_ridge = ridge.predict(Xr_test)
pred_gbr = gbr.predict(Xr_test)

print("Ridge:", eval_regression(ridge, Xr_test, yreg_test)[0])
print("GBR:", eval_regression(gbr, Xr_test, yreg_test)[0])

In [ ]:
# ── TODO: pick churn probabilities and value predictions ────────
p_final = p_rf      # ── TODO: set to your chosen churn model (or calibrated probs) ───
v_final = pred_gbr  # ── TODO: set to your chosen regression model ────────────────────
v_final = np.clip(v_final, 0, None)

call_list = pd.DataFrame({
    "customerID": customer_ids.loc[X_test.index].values,
    "p_churn": p_final,
    "pred_monthly_charges": v_final,
})
call_list["revenue_at_risk"] = call_list["p_churn"] * call_list["pred_monthly_charges"]

call_list = call_list.sort_values("revenue_at_risk", ascending=False)
call_list.head(10)

In [ ]:
# ── Save call list ──────────────────────────────────────────────
out_path = "day5_independent_call_list.csv"
call_list.to_csv(out_path, index=False)
print(f"✅ Saved to {out_path}")
out_path

### GenAI: Interpret model comparison results

In [ ]:
# ── TODO (GenAI): Ask GenAI to interpret model results ──────────
# Uncomment and customize this section to ask for business interpretation

# model_comp = {
#     "Logistic": m_logit,
#     "RandomForest": m_rf,
#     "GradientBoosting": m_gb
# }
# 
# interp_prompt = f"""
# I've trained 3 churn models with these results:
# {json.dumps(model_comp, indent=2)}
# 
# Business context: False negatives (missed churners) cost 5x more than false positives (unnecessary calls).
# 
# In 3-4 sentences, recommend which model to deploy and explain your reasoning based on PR-AUC and business impact.
# """
# 
# response = client.models.generate_content(model=MODEL_ID, contents=interp_prompt)
# print(response.text)
# log_interaction(interp_prompt, response.text)

print("⏳ TODO (optional): Send model comparison to GenAI for interpretation")

---
## Part 4: Manager recommendation (10 min)

✅ **Task:** write a short recommendation.

Include:
- which customers to contact (rule + capacity)
- what metric you optimized and why
- top 3 drivers (from SHAP or feature importance)
- biggest risks (leakage, drift, segment instability)

### TODO: Manager recommendation (write 5–10 bullets)

- **Decision & action:** …
- **Targeting rule:** …
- **Expected impact:** …
- **Top drivers:** …
- **Risks & monitoring:** …

### SHAP Explanation (optional enhancement)

In [ ]:
# ── TODO (GenAI): Ask for SHAP narrative explanations ──────────
# Use SHAP feature importance to explain why the model makes certain predictions

# Extract transformed features for SHAP
# X_test_transformed = preprocess.transform(X_test)
# feature_names = preprocess.get_feature_names_out()
#
# # Pick a high-risk customer
# high_risk_idx = p_rf.argmax()
# shap_prompt = f"""
# Customer {customer_ids.iloc[X_test.index[high_risk_idx]]} has a predicted churn probability of {p_rf[high_risk_idx]:.1%}.
# 
# Top SHAP drivers (features that increase churn risk):
# - Contract=Month-to-month (SHAP=+0.35)
# - MonthlyCharges=120 (SHAP=+0.12)
# - tenure=6 months (SHAP=+0.08)
# 
# Write a short narrative (2-3 sentences) for a retention manager explaining why this customer is at risk.
# """
# 
# response = client.models.generate_content(model=MODEL_ID, contents=shap_prompt)
# print(response.text)
# log_interaction(shap_prompt, response.text)

print("⏳ TODO (optional): Send SHAP insights to GenAI for narrative explanations")

---
## Part 5: GenAI Copilot Log

Record **at least 3** GenAI interactions you used today.

## 🧾 Required: GenAI Copilot Log

Record **at least 3** GenAI interactions you used today.

- **Prompt #1:**
  - What I asked: `...`
  - What changed in my work: `...`
  - What I verified manually: `...`

- **Prompt #2:**
  - What I asked: `...`
  - What changed in my work: `...`
  - What I verified manually: `...`

- **Prompt #3:**
  - What I asked: `...`
  - What changed in my work: `...`
  - What I verified manually: `...`

In [ ]:
# ── Export Prompt Log ───────────────────────────────────────────
if PROMPT_LOG:
    log_df = pd.DataFrame(PROMPT_LOG)
    log_df.to_csv("day5_lab2_prompt_log.csv", index=False)
    print(f"✅ Exported {len(log_df)} prompt log entries to day5_lab2_prompt_log.csv")
    print(log_df)
else:
    print("ℹ️ Prompt log is empty (no GenAI calls yet)")

---
## ✅ Verification Checklist (complete before you finish)

- [ ] Metrics computed on a **holdout test set**
- [ ] Preprocessing inside a **pipeline** (no leakage)
- [ ] One manager-friendly ranking view included (e.g., **lift@10%** or deciles)
- [ ] Revenue-at-Risk call list saved as CSV
- [ ] Your recommendation references **real outputs** (numbers/plots)
- [ ] At least one extension track (A/B/C/D) completed
- [ ] GenAI copilot log has **at least 3** entries
- [ ] Prompt log exported to CSV